# Entrenamiento en Kaggle (modelos clásicos, sin redes neuronales)

Antes de correr:
1. No hace falta **Add Input**: el dataset combinado (`FRANKENSTEIN` en la celda Parámetros) se descarga solo. Si lo adjuntas, se usa el adjunto.
2. **Settings** → Internet: **On** (requiere teléfono verificado). No hace falta GPU: todo corre en CPU.
3. Para correr sin tener la pestaña abierta: **Save Version → Save & Run All**.
4. **Add-ons → Secrets** → activa `GITHUB_TOKEN` para que el modelo se publique solo al final (ver docs/detalles.md, "Publicar el modelo"). Sin él, el notebook entrena igual pero no publica.
5. Las fotos del equipo se bajan solas de la carpeta de Google Drive (`EQUIPO_DRIVE` en la celda de parámetros).
6. Los parámetros de los modelos se ajustan en la celda **Parámetros**, sin tocar el repo.

Este notebook solo clona el repo y llama a `src/`. Los cambios de código van en el repo, no aquí.

In [ ]:
import os, subprocess

REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    r = subprocess.run(["git", "clone", "https://github.com/11909/Clasificador-fruta-verdura.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("No se pudo clonar el repo. Activa Settings → Internet en el panel derecho "
                           "(Kaggle pide verificar tu teléfono) y vuelve a correr esta celda.\n" + r.stderr)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"])  # trae la última versión de main
%cd {REPO}
!pip install -q -r requirements.txt

## Parámetros

Todo lo que se puede ajustar sin tocar el repo. Cambia los valores y corre de aquí hacia abajo.

- **Datos:** `MAX_POR_CLASE` más alto usa más imágenes (mejor, pero más lento).
- **Modelos:** cada entrada de `MODELOS` se compara con validación cruzada y se queda el de mayor f1_macro. Quita las que no quieras o cambia sus `params` (son los de scikit-learn). Opciones de `name`: logistic_regression, svm, knn, naive_bayes, decision_tree, random_forest, gradient_boosting, voting, stacking. `pca` reduce dimensiones antes del modelo (entero = componentes, decimal = varianza a conservar). `grid` prueba varias combinaciones con GridSearchCV.

In [ ]:
# --- Datos ---
FRANKENSTEIN = "mendezrosalesmiguel/frankeinstein"   # dataset combinado; se descarga solo si no está en Add Input
EQUIPO_DRIVE = "https://drive.google.com/drive/folders/1l4Q_VRv__PdaO16_aKlgEwdX50k4FT3S"  # "" = sin fotos del equipo
FRACCION_TEST_EQUIPO = 0.3     # parte de las fotos del equipo que va a test
MAX_POR_CLASE = 150            # imágenes por alimento para entrenar (None = todas; más = más lento)
MAX_POR_CLASE_TEST = None      # None = todo el conjunto de prueba
BALANCEAR_ETIQUETAS = True     # mismo número de frutas que de verduras
EXCLUIR_GRUPOS = []            # ["nueces_semillas"] para sacar nueces y semillas
MINIMO_EQUIPO = 80             # fotos reales del equipo que pide como mínimo cada alimento del plan

# --- Características y validación ---
FEATURES = {"tamano": 64, "tipos": ["color", "hog", "lbp"], "bins_color": 16}
CV = {"tipo": "grupos", "folds": 5}   # tipo: grupos (recomendado) o estratificada
AJUSTAR_UMBRAL = True                 # busca el mejor umbral de verdura en vez de 0.5
SEMILLA = 42

# --- Modelos a comparar ---
MODELOS = [
    {"name": "logistic_regression", "params": {"C": 1.0}},
    {"name": "knn", "params": {"n_neighbors": 7}, "pca": 0.95},
    {"name": "naive_bayes"},
    {"name": "random_forest", "params": {"n_estimators": 300}},
    {"name": "gradient_boosting", "params": {"max_iter": 300, "learning_rate": 0.1}},
    {"name": "svm", "params": {"kernel": "rbf", "C": 10, "gamma": "scale"}},
    {"name": "voting", "params": {"estimadores": {
        "svm": {"kernel": "rbf", "C": 10, "gamma": "scale"},
        "random_forest": {"n_estimators": 300},
        "gradient_boosting": {"max_iter": 300},
    }}},
    # Ejemplo con búsqueda de hiperparámetros:
    # {"name": "random_forest", "grid": {"randomforestclassifier__max_depth": [None, 20],
    #                                    "randomforestclassifier__n_estimators": [200, 400]}},
]

Localiza el dataset combinado de Kaggle, baja las fotos del equipo de Drive y genera la config de esta corrida con los parámetros de arriba.

In [ ]:
import glob, os, yaml

# Dataset combinado del equipo (tiene manifest.csv): el de Add Input o, si no está, se descarga.
manifest = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
if not manifest and FRANKENSTEIN:
    try:
        import kagglehub
        ruta = kagglehub.dataset_download(FRANKENSTEIN)
        manifest = glob.glob(f"{ruta}/**/manifest.csv", recursive=True)
    except Exception as e:
        print(f"No pude descargar {FRANKENSTEIN} ({e}). Pide a Miguel Angel acceso al dataset.")
if manifest:
    base = os.path.dirname(manifest[0])
    train_dirs, test_dirs = [f"{base}/train"], [f"{base}/test"]
else:
    train_dirs = sorted(glob.glob("/kaggle/input/**/Training", recursive=True))
    test_dirs = sorted(glob.glob("/kaggle/input/**/Test", recursive=True))
if not train_dirs or not test_dirs:
    print("Contenido de /kaggle/input:", glob.glob("/kaggle/input/*/*")[:20])
    raise FileNotFoundError("No encontré las carpetas de imágenes. Activa Settings → Internet o adjunta "
                            "el dataset combinado con Add Input y vuelve a correr.")
print("Kaggle train:", train_dirs[0])
print("Kaggle test:", test_dirs[0])

# Fotos del equipo desde Google Drive
equipo_dirs = []
if EQUIPO_DRIVE:
    from src.equipo import descargar_drive
    try:
        equipo_dirs = descargar_drive(EQUIPO_DRIVE)
    except Exception as e:
        raise RuntimeError("No pude bajar la carpeta de Drive. Revisa que esté compartida como "
                           "'Cualquier persona con el enlace' y que Settings → Internet esté activo. "
                           "Si Drive dice que hubo demasiados accesos, espera un rato y vuelve a correr.") from e
    print("Fotos del equipo:", equipo_dirs)
else:
    print("Sin fotos del equipo (EQUIPO_DRIVE vacío).")

cfg = yaml.safe_load(open("configs/baseline.yaml"))
cfg["data"].update({
    "train_dirs": train_dirs[:1], "test_dirs": test_dirs[:1], "equipo_dir": equipo_dirs or None,
    "fraccion_test_equipo": FRACCION_TEST_EQUIPO, "max_por_clase": MAX_POR_CLASE,
    "max_por_clase_test": MAX_POR_CLASE_TEST, "balancear_etiquetas": BALANCEAR_ETIQUETAS,
    "excluir_grupos": EXCLUIR_GRUPOS,
})
cfg.update({"features": FEATURES, "cv": CV, "ajustar_umbral": AJUSTAR_UMBRAL, "seed": SEMILLA, "models": MODELOS})
yaml.safe_dump(cfg, open("configs/run.yaml", "w"), allow_unicode=True)

Cuenta las imágenes de cada fuente y escribe `faltantes.txt` (en `/kaggle/working/outputs/`): cuántas fotos le faltan al equipo por alimento para que cada uno llegue a `MAX_POR_CLASE` en train, con al menos `MINIMO_EQUIPO` fotos reales. También revisa que todas las carpetas tengan etiqueta en `data/mapping.csv`.

In [ ]:
from pathlib import Path
from src.dataset import dividir_equipo, listar_muestras
from src.equipo import contar_por_alimento, escribir_faltantes, faltantes, leer_plan

kaggle = contar_por_alimento(listar_muestras(train_dirs[:1], excluir_grupos=EXCLUIR_GRUPOS))
equipo_muestras = []
for carpeta in equipo_dirs:
    a, b = dividir_equipo(carpeta, FRACCION_TEST_EQUIPO, SEMILLA, EXCLUIR_GRUPOS)
    equipo_muestras += a + b
equipo_conteo = contar_por_alimento(equipo_muestras)
print(f"Kaggle: {sum(kaggle.values())} imágenes de train en {len(kaggle)} alimentos")
print(f"Equipo: {len(equipo_muestras)} fotos en {len(equipo_conteo)} alimentos")

salida = Path(cfg["output_dir"]); salida.mkdir(parents=True, exist_ok=True)
filas = faltantes(kaggle, equipo_conteo, MAX_POR_CLASE or 150, FRACCION_TEST_EQUIPO, MINIMO_EQUIPO, leer_plan())
print(escribir_faltantes(filas, salida / "faltantes.txt"))

Extrae características, compara los modelos con validación cruzada, reentrena el mejor y lo evalúa en test.

In [ ]:
!python -m src.train --config configs/run.yaml

Resultados: comparación de modelos (`resultados_cv.csv`), métricas en test y el modelo guardado (`modelo.joblib`) en `/kaggle/working/outputs/`.

In [ ]:
import csv, json
for fila in csv.DictReader(open("/kaggle/working/outputs/resultados_cv.csv")):
    print(f"{fila['modelo']:20} f1_macro={fila['f1_macro']}  ±{fila['f1_macro_std']}  acc_balanceada={fila['balanced_accuracy']}  ({fila['segundos']} s)")
print(json.dumps(json.load(open("/kaggle/working/outputs/test_metrics.json")), indent=2))

## Revisar y publicar el modelo

Prueba rápida del modelo recién entrenado con las entradas de ejemplo. Después, la última celda lo publica sola como Release de GitHub, siempre que supere en accuracy balanceada de test al modelo publicado. Desde ese momento `notebooks/predecir.ipynb`, `python -m src.predict` y la interfaz usan el modelo nuevo.

In [ ]:
from src.clasificador import Clasificador

clf = Clasificador("/kaggle/working/outputs/modelo.joblib")
print(clf.descripcion)
for r in clf.clasificar_lote("entradas/ejemplos.txt"):
    print(r.get("etiqueta", "ERROR"), f"{r.get('confianza', 0):.0%}", r["entrada"].split("/")[-2:], r.get("error", ""))

In [ ]:
import os
from src.publicar import publicar

PUBLICAR = True   # False para entrenar sin publicar
FORZAR = False    # True publica aunque no supere al modelo publicado

token = os.environ.get("GITHUB_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None

if not PUBLICAR:
    print("Publicación desactivada (PUBLICAR = False).")
elif not token:
    print("No encontré el secreto GITHUB_TOKEN, así que el modelo no se publicó. "
          "Agrégalo en Add-ons → Secrets (ver docs/detalles.md, \"Publicar el modelo\") y corre esta celda otra vez.")
else:
    publicar("/kaggle/working/outputs", token, forzar=FORZAR)